# Lab 9: Text as Data

**ECON5129 Statistical Machine Learning** &middot; Adam Smith Business School, University of Glasgow

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/korobilis/ECON5129-labs/blob/main/solutions/lab09_text_as_data_solutions.ipynb)

This lab accompanies Lecture 9. By the end of the session you should be able to:

1. Turn a corpus of documents into a numerical matrix, and state what each preprocessing choice throws away.
2. Build term frequency and inverse document frequency weights by hand and explain what the weighting corrects for.
3. Measure similarity between documents and track how the language of monetary policy changes over time.
4. Construct a dictionary-based sentiment index and validate it against something observable.
5. Estimate word embeddings from co-occurrence counts using pointwise mutual information and a singular value decomposition.
6. Fit a topic model and interpret its output without over-claiming.

The corpus is the transcripts of Federal Open Market Committee meetings from 1977 to 2008: 268 meetings, released with a five-year lag, covering the Volcker disinflation, the Greenspan years and the approach to the financial crisis.

**Plan for the session**

| Time | Part |
|---|---|
| 0:00 - 0:20 | Part 1. From documents to a matrix |
| 0:20 - 0:40 | Part 2. Weighting and similarity |
| 0:40 - 1:05 | Part 3. Sentiment |
| 1:05 - 1:30 | Part 4. Word embeddings |
| 1:30 - 2:00 | Part 5. Topic models and measurement |

## Setup

In [ ]:
import os
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/korobilis/ECON5129-labs/main"

if not os.path.exists("econ5129_utils.py"):
    urllib.request.urlretrieve(f"{REPO_RAW}/econ5129_utils.py", "econ5129_utils.py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import econ5129_utils as e5

e5.set_style()
rng = np.random.default_rng(5129)

## Part 1. From documents to a matrix

Every statistical method in this course takes a matrix. Text is not a matrix, so the first decision in any text project is how to make one, and every choice made along the way destroys information deliberately.

In [ ]:
meetings = e5.load_fomc_meetings()
print(f"{len(meetings)} meetings from {meetings['date'].min():%Y-%m} to {meetings['date'].max():%Y-%m}")
print(meetings[["date", "chair", "n_speakers", "n_words"]].head())

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(meetings["date"], meetings["n_words"] / 1000, color=e5.COLORS[0], lw=1.0)
ax.set_ylabel("thousands of words")
ax.set_title("Length of FOMC meeting transcripts")
plt.show()

**Tokenisation** splits a document into words, lowercases them, and drops punctuation and very common words. `e5.tokenise` does exactly that, and it is worth reading its source: it is six lines and each one is a modelling assumption.

In [ ]:
example = meetings["text"].iloc[100][:400]
print("Raw text:\n", example, "\n")
print("Tokens:\n", e5.tokenise(example)[:25])

Word frequencies in any natural corpus follow Zipf's law: the $r$th most common word appears roughly proportionally to $1/r$. A handful of words carry most of the mass and a long tail appears once or twice.

In [ ]:
from collections import Counter

all_tokens = [t for doc in meetings["text"] for t in e5.tokenise(doc)]
counts = Counter(all_tokens)
frequencies = np.array(sorted(counts.values(), reverse=True))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].loglog(np.arange(1, len(frequencies) + 1), frequencies, color=e5.COLORS[0])
axes[0].set_xlabel("rank")
axes[0].set_ylabel("frequency")
axes[0].set_title("Zipf's law")

top = counts.most_common(15)
axes[1].barh([w for w, _ in top][::-1], [c for _, c in top][::-1], color=e5.COLORS[1])
axes[1].set_xlabel("count")
axes[1].set_title("Fifteen most frequent tokens")
fig.tight_layout()
plt.show()

print(f"vocabulary size: {len(counts):,}")
print(f"words appearing once: {sum(1 for c in counts.values() if c == 1):,}")

## Part 2. Weighting and similarity

The document-term matrix counts how often each term appears in each document. Terms appearing in fewer than five meetings are dropped, since they cannot support any comparison. Terms appearing in every meeting are deliberately kept: in a corpus of 268 meetings of the same committee, words like "inflation" appear everywhere, and discarding them would remove exactly the vocabulary we want to study. Inverse document frequency handles them instead by downweighting rather than deleting:

$$ \text{tfidf}_{dv} = \text{tf}_{dv} \times \log\frac{D}{d_v}, $$

where $\text{tf}_{dv}$ is the count of term $v$ in document $d$, $D$ the number of documents, and $d_v$ the number of documents containing $v$.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vec = CountVectorizer(tokenizer=e5.tokenise, lowercase=False,
                            min_df=5, token_pattern=None)
counts_matrix = count_vec.fit_transform(meetings["text"])
vocab = np.array(count_vec.get_feature_names_out())

print(f"document-term matrix: {counts_matrix.shape[0]} documents by {counts_matrix.shape[1]} terms")
print(f"density: {counts_matrix.nnz / np.prod(counts_matrix.shape):.1%}")

In [ ]:
tf = counts_matrix.toarray().astype(float)
document_frequency = (tf > 0).sum(axis=0)
idf = np.log(len(meetings) / document_frequency)
tfidf_manual = tf * idf

tfidf_vec = TfidfVectorizer(tokenizer=e5.tokenise, lowercase=False, min_df=5,
                            token_pattern=None, norm=None, smooth_idf=False)
tfidf_sklearn = tfidf_vec.fit_transform(meetings["text"]).toarray()

print("Terms with the highest inverse document frequency, meaning the most distinctive:")
print(", ".join(vocab[np.argsort(-idf)[:12]]))
print("\nTerms with the lowest, meaning they appear nearly everywhere:")
print(", ".join(vocab[np.argsort(idf)[:12]]))

### Cosine similarity

Two documents are compared by the angle between their vectors, which ignores length,

$$ \cos\theta_{ij} = \frac{\mathbf{x}_i'\mathbf{x}_j}{\|\mathbf{x}_i\|\,\|\mathbf{x}_j\|}. $$

In [ ]:
from sklearn.preprocessing import normalize

normalised = normalize(tfidf_manual)
similarity = normalised @ normalised.T

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
im = axes[0].imshow(similarity, cmap="viridis", origin="lower")
ticks = np.linspace(0, len(meetings) - 1, 6).astype(int)
axes[0].set_xticks(ticks, [f"{meetings['date'].iloc[t]:%Y}" for t in ticks])
axes[0].set_yticks(ticks, [f"{meetings['date'].iloc[t]:%Y}" for t in ticks])
axes[0].set_title("Similarity between all pairs of meetings")
fig.colorbar(im, ax=axes[0], fraction=0.046)

consecutive = np.array([similarity[i, i + 1] for i in range(len(meetings) - 1)])
axes[1].plot(meetings["date"].iloc[1:], consecutive, color=e5.COLORS[0], lw=1.0)
axes[1].set_title("Similarity between consecutive meetings")
fig.tight_layout()
plt.show()

### Exercise 1

Language changes slowly, then quickly.

1. For each meeting, compute its average similarity to all meetings held more than five years earlier.
2. Plot that series over time.
3. Identify the three periods in which the language of the committee moved furthest from its own past, and relate them to what was happening in the economy.

In [ ]:
dates = meetings["date"].to_numpy()
years = meetings["date"].dt.year.to_numpy()

distance_to_past = np.full(len(meetings), np.nan)
for i in range(len(meetings)):
    old = years < years[i] - 5
    if old.sum() > 5:
        distance_to_past[i] = similarity[i, old].mean()

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(meetings["date"], distance_to_past, color=e5.COLORS[0])
ax.set_ylabel("average similarity to meetings 5+ years earlier")
ax.set_title("How far the committee's language has drifted from its own past")
plt.show()

valid = ~np.isnan(distance_to_past)
lowest = np.argsort(np.where(valid, distance_to_past, np.inf))[:3]
print("Meetings least similar to their own past:")
for i in lowest:
    print(f"  {meetings['date'].iloc[i]:%Y-%m}, chaired by {meetings['chair'].iloc[i]}")

## Part 3. Sentiment

The dictionary approach counts words from predefined lists. It is transparent, reproducible, requires no training data, and is crude. For central bank communication the relevant dimension is not positive against negative but hawkish against dovish, so the course dictionary in `e5.TONE_DICTIONARY` contains both pairs.

The tone index is the difference in shares,

$$ \text{tone}_d = \frac{\#\{\text{hawkish words in } d\} - \#\{\text{dovish words in } d\}}{\#\{\text{words in } d\}}. $$

In [ ]:
def dictionary_score(tokens, positive_set, negative_set):
    """Net share of words falling in two opposing dictionaries."""
    n = len(tokens)
    if n == 0:
        return 0.0
    positive = sum(1 for t in tokens if t in positive_set)
    negative = sum(1 for t in tokens if t in negative_set)
    return (positive - negative) / n


tokens_by_meeting = [e5.tokenise(doc) for doc in meetings["text"]]

meetings["tone"] = [dictionary_score(t, e5.TONE_DICTIONARY["hawkish"],
                                     e5.TONE_DICTIONARY["dovish"]) for t in tokens_by_meeting]
meetings["sentiment"] = [dictionary_score(t, e5.TONE_DICTIONARY["positive"],
                                          e5.TONE_DICTIONARY["negative"]) for t in tokens_by_meeting]

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(meetings["date"], meetings["tone"], color=e5.COLORS[0], lw=1.0)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_ylabel("hawkish minus dovish")

axes[1].plot(meetings["date"], meetings["sentiment"], color=e5.COLORS[2], lw=1.0)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_ylabel("positive minus negative")
fig.suptitle("Two dictionary indices of FOMC deliberation")
fig.tight_layout()
plt.show()

### Validation

An index that has never been checked against anything observable is an opinion with a number attached. The natural test is whether the hawkish index moves with the policy rate the committee actually set.

In [ ]:
data, codes = e5.load_fredmd()
monthly = data[["FEDFUNDS", "UNRATE", "CPIAUCSL"]].copy()
monthly["inflation"] = 1200 * np.log(monthly["CPIAUCSL"]).diff()

aligned = pd.merge_asof(
    meetings[["date", "tone", "sentiment"]].sort_values("date"),
    monthly.reset_index().rename(columns={"date": "month"}).sort_values("month"),
    left_on="date", right_on="month", direction="backward",
).dropna()

aligned["rate_change"] = aligned["FEDFUNDS"].diff()

print(aligned[["tone", "sentiment", "FEDFUNDS", "inflation", "UNRATE", "rate_change"]]
      .corr().round(3)[["tone", "sentiment"]])

### Exercise 2

Dictionary indices are sensitive to the dictionary, which is precisely why they must be probed rather than trusted.

1. Remove the word `unemployment` from the dovish list, since it names a variable rather than expressing a stance, and recompute the tone index.
2. Plot the two versions together and compute their correlation.
3. Construct a third version using only the ten most frequent words from each list, and compare again.

How much of the index is driven by a handful of common words?

In [ ]:
hawkish = set(e5.TONE_DICTIONARY["hawkish"])
dovish_trimmed = set(e5.TONE_DICTIONARY["dovish"]) - {"unemployment"}

tone_trimmed = np.array([dictionary_score(t, hawkish, dovish_trimmed)
                         for t in tokens_by_meeting])

top_hawkish = {w for w, _ in Counter(
    t for tokens in tokens_by_meeting for t in tokens if t in hawkish).most_common(10)}
top_dovish = {w for w, _ in Counter(
    t for tokens in tokens_by_meeting for t in tokens
    if t in e5.TONE_DICTIONARY["dovish"]).most_common(10)}
tone_top10 = np.array([dictionary_score(t, top_hawkish, top_dovish)
                       for t in tokens_by_meeting])

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(meetings["date"], meetings["tone"], label="full dictionary")
ax.plot(meetings["date"], tone_trimmed, label="without 'unemployment'")
ax.plot(meetings["date"], tone_top10, label="ten most frequent words only")
ax.axhline(0, color="black", lw=0.8)
ax.legend(fontsize=9)
ax.set_title("How much does the dictionary choice matter?")
plt.show()

print(pd.DataFrame({
    "full": meetings["tone"], "trimmed": tone_trimmed, "top ten": tone_top10
}).corr().round(3))

## Part 4. Word embeddings

Dictionaries treat words as unrelated symbols. Embeddings place them in a vector space where distance means something, and they can be estimated from the corpus itself.

The classical construction needs no neural network. Count how often words co-occur within a window, convert counts to **positive pointwise mutual information**,

$$ \text{PPMI}(v, w) = \max\left\{0, \; \log \frac{p(v, w)}{p(v)p(w)}\right\}, $$

and take a truncated singular value decomposition. The rows of the result are word vectors.

In [ ]:
def cooccurrence_matrix(token_lists, vocabulary, window=5):
    """Symmetric co-occurrence counts within a fixed word window."""
    index = {w: i for i, w in enumerate(vocabulary)}
    C = np.zeros((len(vocabulary), len(vocabulary)))
    for tokens in token_lists:
        ids = [index[t] for t in tokens if t in index]
        for position, centre in enumerate(ids):
            lo = max(0, position - window)
            hi = min(len(ids), position + window + 1)
            for other in ids[lo:position] + ids[position + 1:hi]:
                C[centre, other] += 1.0
    return C


vocabulary = [w for w, _ in counts.most_common(600)]
C = cooccurrence_matrix(tokens_by_meeting, vocabulary, window=5)

total = C.sum()
p_joint = C / total
p_word = C.sum(axis=1) / total
with np.errstate(divide="ignore", invalid="ignore"):
    pmi = np.log(p_joint / np.outer(p_word, p_word))
ppmi = np.nan_to_num(np.maximum(pmi, 0.0), nan=0.0, posinf=0.0, neginf=0.0)

U_emb, d_emb, _ = np.linalg.svd(ppmi, full_matrices=False)
embeddings = U_emb[:, :50] * d_emb[:50]
embeddings = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)

word_index = {w: i for i, w in enumerate(vocabulary)}


def nearest_words(word, k=8):
    """Words whose embeddings are closest in cosine distance."""
    if word not in word_index:
        return []
    scores = embeddings @ embeddings[word_index[word]]
    order = np.argsort(-scores)[1:k + 1]
    return [(vocabulary[i], float(scores[i])) for i in order]


for probe in ["inflation", "unemployment", "credit"]:
    neighbours = nearest_words(probe)
    if neighbours:
        print(f"{probe:14s}: " + ", ".join(w for w, _ in neighbours))

## Part 5. Topic models and measurement

A topic model represents each document as a mixture over topics and each topic as a distribution over words. Latent semantic analysis does this with a singular value decomposition; latent Dirichlet allocation does it with an explicit probability model, which gives non-negative and more interpretable components.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD

n_topics = 6
lda = LatentDirichletAllocation(n_components=n_topics, learning_method="batch",
                                max_iter=25, random_state=0)
document_topics = lda.fit_transform(counts_matrix)

for k, component in enumerate(lda.components_):
    top_words = vocab[np.argsort(-component)[:10]]
    print(f"topic {k}: " + ", ".join(top_words))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
shares = pd.DataFrame(document_topics, columns=[f"topic {k}" for k in range(n_topics)])
shares["date"] = meetings["date"].to_numpy()
smoothed = shares.set_index("date").rolling(8, min_periods=1).mean()

ax.stackplot(smoothed.index, smoothed.T.to_numpy(), labels=smoothed.columns, alpha=0.85)
ax.set_ylim(0, 1)
ax.set_title("Topic shares of FOMC deliberation, smoothed over eight meetings")
ax.legend(loc="upper left", ncol=3, fontsize=8)
plt.show()

### Exercise 3

Topics are only useful if they measure something.

1. Align the topic shares with monthly macroeconomic data using `pd.merge_asof`, as in Part 3.
2. Compute the correlation of each topic share with inflation, the unemployment rate and the federal funds rate.
3. Identify which topic behaves most like an inflation-concern index and which most like a labour-market index, and plot each against its macroeconomic counterpart.

Remember that a correlation between a topic share and a macroeconomic variable is a measurement claim, not a causal one.

In [ ]:
topic_frame = shares.copy()
merged = pd.merge_asof(
    topic_frame.sort_values("date"),
    monthly.reset_index().rename(columns={"date": "month"}).sort_values("month"),
    left_on="date", right_on="month", direction="backward",
).dropna()

topic_cols = [c for c in merged.columns if c.startswith("topic")]
correlations = merged[topic_cols + ["inflation", "UNRATE", "FEDFUNDS"]].corr()
print(correlations.loc[topic_cols, ["inflation", "UNRATE", "FEDFUNDS"]].round(3))

inflation_topic = correlations.loc[topic_cols, "inflation"].idxmax()
labour_topic = correlations.loc[topic_cols, "UNRATE"].idxmax()

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(merged["date"], merged[inflation_topic], color=e5.COLORS[0], label=inflation_topic)
axes[0].twinx().plot(merged["date"], merged["inflation"], color=e5.COLORS[2], lw=0.9)
axes[0].set_title(f"{inflation_topic} against CPI inflation")

axes[1].plot(merged["date"], merged[labour_topic], color=e5.COLORS[0], label=labour_topic)
axes[1].twinx().plot(merged["date"], merged["UNRATE"], color=e5.COLORS[2], lw=0.9)
axes[1].set_title(f"{labour_topic} against the unemployment rate")
fig.tight_layout()
plt.show()

### Exercise 4

Compare the two ways of reducing the document-term matrix.

1. Fit `TruncatedSVD` with six components to the tf-idf matrix.
2. Print the ten highest-loading words for each component.
3. Contrast them with the latent Dirichlet allocation topics above.

Which is easier to interpret, and what is the source of the difference?

In [ ]:
svd = TruncatedSVD(n_components=6, random_state=0)
lsa_scores = svd.fit_transform(tfidf_manual)

for k, component in enumerate(svd.components_):
    positive = vocab[np.argsort(-component)[:8]]
    negative = vocab[np.argsort(component)[:5]]
    print(f"component {k}: +({', '.join(positive)})  -({', '.join(negative)})")

print(f"\nvariance explained by six components: {svd.explained_variance_ratio_.sum():.1%}")

## Take-home challenges

1. **Speaker-level analysis.** Using the utterance-level file, compute the tone index separately for the chair and for the other participants at each meeting. Does the chair speak more hawkishly than the committee average, and does the gap change across chairs?

2. **N-grams.** Rebuild the document-term matrix using bigrams as well as single words. Which two-word phrases have the highest inverse document frequency, and do any of them capture stances that single words miss?

3. **Choosing the number of topics.** Fit latent Dirichlet allocation for two to fifteen topics and plot the held-out perplexity. Does the criterion pick a number you would have chosen by reading the topics?

4. **Dictionary versus supervised.** Label fifty meetings by hand as hawkish, neutral or dovish from their topic shares and tone, then train a classifier on tf-idf features to reproduce your labels. Report cross-validated accuracy and inspect the words the classifier relies on.

---

**Next**: Lecture 10 replaces counting with representation learning. Lab 10 implements attention from scratch, uses a pretrained language model, and asks whether any of it improves a macroeconomic forecast.